<a href="https://colab.research.google.com/github/Wai-Fun/Healthcare-Data-Analytics/blob/main/Building_and_Querying_a_Relational_Healthcare_Database_Using_SQL_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building and Querying a Relational Healthcare Database Using SQL



## Introduction

Healthcare organizations rely on relational databases to store and connect information about patients, visits, diagnoses, billing, and lab results.

Instead of keeping all data in a single large table, relational databases organize information into smaller, linked tables that:

- Reduce errors and duplication  
- Improve consistency  
- Make querying much more efficient  

The project include:
- Designed a relational healthcare database with normalized tables for patients, visits (billing), and lab results
- Applied primary and foreign keys to link related records and maintain referential integrity
- Used SQL to retrieve and combine information across tables with SELECT, WHERE, ORDER BY, and JOIN

These patterns reflect how relational databases support efficient, accurate healthcare data management in real-world clinical and operational systems.

## Setup: Create a SQLite Database in Python

This notebook uses **SQLite** via Python's built-in `sqlite3` module.


In [98]:
# Import libraries
import sqlite3
import pandas as pd
from pathlib import Path

# Create / connect to a local SQLite database file
db_path = Path("healthcare_lab.db")  # create a file namehealthcare_lab.db; if it exist, open it.
conn = sqlite3.connect(db_path)  # Open that file as a database; if it does not exist yet, python creates it automatically
cursor = conn.cursor()  #

print(f"Connected to SQLite database at: {db_path.resolve()}")

def run_sql(sql: str):  # run sql commands and commited
    """Execute one or more SQL statements (no result returned)."""
    cursor.executescript(sql)
    conn.commit()

def show_query(sql: str):  # show the dataframe
    """Run a SELECT query and display results as a pandas DataFrame."""
    print("SQL query:")
    print(sql.strip())
    df = pd.read_sql_query(sql, conn)
    # display(df) -- # Redundant line: comment out this line of code that will only display the dataframe
    return df # keep this line that dispplay the dataframe and then make the dataframe available for use in the later stage.

Connected to SQLite database at: /content/healthcare_lab.db


## Task 1: Create the Healthcare Database Tables

Create three tables:

1. `Patients` – one row per patient  
2. `Visits` – one row per visit (billing record)  
3. `LabTestResults` – one row per lab result, linked to both patient and visit

### 1.1 Create the `Patients` Table (SQLite)

In [99]:
patients_sql = '''
DROP TABLE IF EXISTS Patients;

CREATE TABLE Patients (
    PatientID TEXT PRIMARY KEY,
    Name TEXT,
    DOB TEXT NOT NULL,
    Gender TEXT,
    Address TEXT
);
'''

run_sql(patients_sql)
print("Patients table created.")


Patients table created.


### 1.2 Create the `Visits` Table (SQLite)

In [100]:
visits_sql = '''
DROP TABLE IF EXISTS Visits;

CREATE TABLE Visits (
    VisitID TEXT PRIMARY KEY,
    PatientID TEXT NOT NULL,
    VisitDate TEXT,
    ProcedureCode TEXT,
    ProcedureDescription TEXT,
    Charge REAL,
    Payer TEXT,
    FOREIGN KEY (PatientID) REFERENCES Patients(PatientID)
);
'''

run_sql(visits_sql)
print("Visits table created.")

Visits table created.


### 1.3 Create the `LabTestResults` Table (SQLite)

In [101]:
lab_sql = '''
DROP TABLE IF EXISTS LabTestResults;

CREATE TABLE LabTestResults (
    LabResultID TEXT PRIMARY KEY,
    VisitID TEXT NOT NULL,
    PatientID TEXT NOT NULL,
    LabTest TEXT,
    ResultValue TEXT,
    ResultDate TEXT,
    FOREIGN KEY (VisitID) REFERENCES Visits(VisitID),
    FOREIGN KEY (PatientID) REFERENCES Patients(PatientID)
);
'''

run_sql(lab_sql)
print("LabTestResults table created.")

LabTestResults table created.


## Task 2: Insert Data into Tables
Add the available data into the three tables.

### 2.1 Insert Data into `Patients`

In [102]:
insert_patients_sql = '''
INSERT INTO Patients (PatientID, Name, DOB, Gender, Address) VALUES
('P001', 'John Smith', '1985-02-14', 'Male', '1245 Willow Creek Drive, Raleigh, NC 27603'),
('P002', 'Maria Lopez', '1990-11-03', 'Female', '892 North Ridge Avenue, Durham, NC 27701'),
('P003', 'Tim Chen', '1978-06-22', 'Male', '2108 Maplewood Lane, Chapel Hill, NC 27514'),
('P004', 'Aisha Khan', '1990-09-09', 'Female', '55 Harbor Point Road, Charlotte, NC 28202');
'''

run_sql(insert_patients_sql)
print("Sample patients inserted.")

# Verify
show_query("SELECT * FROM Patients;")

Sample patients inserted.
SQL query:
SELECT * FROM Patients;


,PatientID,Name,DOB,Gender,Address
0,P001,John Smith,1985-02-14,Male,"1245 Willow Creek Drive, Raleigh, NC 27603"
1,P002,Maria Lopez,1990-11-03,Female,"892 North Ridge Avenue, Durham, NC 27701"
2,P003,Tim Chen,1978-06-22,Male,"2108 Maplewood Lane, Chapel Hill, NC 27514"
3,P004,Aisha Khan,1990-09-09,Female,"55 Harbor Point Road, Charlotte, NC 28202"


### 2.2 Insert Data into `Visits` (Billing)

In [103]:
insert_visits_sql = '''
INSERT INTO Visits (VisitID, PatientID, ProcedureCode, ProcedureDescription, Charge, Payer, VisitDate) VALUES
('V001', 'P001', '85025', 'Complete Blood Count (CBC)', 45.00, 'Aetna', '2024-01-12'),
('V002', 'P002', '99213', 'Office Visit – Established Patient', 90.00, 'BlueCross', '2024-01-15'),
('V003', 'P003', '36415', 'Venipuncture', 15.00, 'Medicare', '2024-01-20'),
('V004', 'P001', '80053', 'Comprehensive Metabolic Panel (CMP)', 55.00, 'Aetna', '2024-02-01'),
('V005', 'P004', '87880', 'Rapid Strep Test', 30.00, 'Cigna', '2024-02-05');
'''

run_sql(insert_visits_sql)
print("Sample visits inserted.")

# Verify
show_query("SELECT * FROM Visits;")

Sample visits inserted.
SQL query:
SELECT * FROM Visits;


,VisitID,PatientID,VisitDate,ProcedureCode,ProcedureDescription,Charge,Payer
0,V001,P001,2024-01-12,85025,Complete Blood Count (CBC),45.0,Aetna
1,V002,P002,2024-01-15,99213,Office Visit – Established Patient,90.0,BlueCross
2,V003,P003,2024-01-20,36415,Venipuncture,15.0,Medicare
3,V004,P001,2024-02-01,80053,Comprehensive Metabolic Panel (CMP),55.0,Aetna
4,V005,P004,2024-02-05,87880,Rapid Strep Test,30.0,Cigna


### 2.3 Insert Data into `LabTestResults`

In [104]:
insert_labs_sql = '''
INSERT INTO LabTestResults (LabResultID, VisitID, PatientID, LabTest, ResultValue, ResultDate) VALUES
('LR001', 'V001', 'P001', 'CBC – Hemoglobin', '13.5 g/dL', '2024-01-12'),
('LR002', 'V001', 'P001', 'CBC – WBC Count', '7.8 x10^3/uL', '2024-01-12'),
('LR003', 'V003', 'P003', 'Venipuncture – Sample Drawn', 'Completed', '2024-01-20'),
('LR004', 'V004', 'P001', 'CMP – Glucose', '92 mg/dL', '2024-02-01'),
('LR005', 'V005', 'P004', 'Rapid Strep Test', 'Negative', '2024-02-05');
'''

run_sql(insert_labs_sql)
print("Sample lab test results inserted.")

# Verify
show_query("SELECT * FROM LabTestResults;")


Sample lab test results inserted.
SQL query:
SELECT * FROM LabTestResults;


,LabResultID,VisitID,PatientID,LabTest,ResultValue,ResultDate
0,LR001,V001,P001,CBC – Hemoglobin,13.5 g/dL,2024-01-12
1,LR002,V001,P001,CBC – WBC Count,7.8 x10^3/uL,2024-01-12
2,LR003,V003,P003,Venipuncture – Sample Drawn,Completed,2024-01-20
3,LR004,V004,P001,CMP – Glucose,92 mg/dL,2024-02-01
4,LR005,V005,P004,Rapid Strep Test,Negative,2024-02-05


## Task 3: Query the Database



### Query 1: Which patients had Complete Blood Count (CBC) procedures?

In [105]:
# Query 1:  Which patients had Complete Blood Count (CBC) procedures? -- using nested query/ subquery
query1 = '''
SELECT Name, DOB
FROM Patients
WHERE PatientID IN (
    SELECT PatientID
    FROM Visits
    WHERE ProcedureDescription LIKE '%Complete Blood Count (CBC)%'
);
'''

show_query(query1)


SQL query:
SELECT Name, DOB
FROM Patients
WHERE PatientID IN (
    SELECT PatientID
    FROM Visits
    WHERE ProcedureDescription LIKE '%Complete Blood Count (CBC)%'
);


,Name,DOB
0,John Smith,1985-02-14


In [106]:
# Alternative # Query 1:  Which patients had Complete Blood Count (CBC) procedures? -- using JOIN

query1 ='''
SELECT p.Name, p.DOB
FROM Patients p
JOIN Visits v ON p.PatientID = v.PatientID
WHERE v.ProcedureDescription LIKE '%Complete Blood Count (CBC)%';
'''
show_query(query1)

SQL query:
SELECT p.Name, p.DOB
FROM Patients p
JOIN Visits v ON p.PatientID = v.PatientID
WHERE v.ProcedureDescription LIKE '%Complete Blood Count (CBC)%';


,Name,DOB
0,John Smith,1985-02-14


### Query 2: Sort all patients alphabetically by name

In [107]:
# Query 2: Sort all patients alphabetically by name
query2 = '''
SELECT Name, DOB, Gender
FROM Patients
ORDER BY Name;
'''

show_query(query2)

SQL query:
SELECT Name, DOB, Gender
FROM Patients
ORDER BY Name;


,Name,DOB,Gender
0,Aisha Khan,1990-09-09,Female
1,John Smith,1985-02-14,Male
2,Maria Lopez,1990-11-03,Female
3,Tim Chen,1978-06-22,Male


### Query 3: Show each patient with their visit dates (LEFT JOIN)

In [108]:
# Query 3: Show each patient with their visit dates (LEFT JOIN)
query3 = '''
SELECT
    Patients.Name,
    Visits.VisitDate
FROM Patients
LEFT JOIN Visits
    ON Patients.PatientID = Visits.PatientID
ORDER BY Patients.Name, Visits.VisitDate;
'''

show_query(query3)

SQL query:
SELECT
    Patients.Name,
    Visits.VisitDate
FROM Patients
LEFT JOIN Visits
    ON Patients.PatientID = Visits.PatientID
ORDER BY Patients.Name, Visits.VisitDate;


,Name,VisitDate
0,Aisha Khan,2024-02-05
1,John Smith,2024-01-12
2,John Smith,2024-02-01
3,Maria Lopez,2024-01-15
4,Tim Chen,2024-01-20


## Query 4: Inspect Table Schemas

Use SQLite PRAGMA commands to inspect the structure of each table.

In [109]:
# Use SQLite PRAGMA commands to inspect the structure of table Patients
show_query("PRAGMA table_info(Patients);")


SQL query:
PRAGMA table_info(Patients);


,cid,name,type,notnull,dflt_value,pk
0,0,PatientID,TEXT,0,None,1
1,1,Name,TEXT,0,None,0
2,2,DOB,TEXT,1,None,0
3,3,Gender,TEXT,0,None,0
4,4,Address,TEXT,0,None,0


In [110]:
# Use SQLite PRAGMA commands to inspect the structure of table Visits
show_query("PRAGMA table_info(Visits);")


SQL query:
PRAGMA table_info(Visits);


,cid,name,type,notnull,dflt_value,pk
0,0,VisitID,TEXT,0,None,1
1,1,PatientID,TEXT,1,None,0
2,2,VisitDate,TEXT,0,None,0
3,3,ProcedureCode,TEXT,0,None,0
4,4,ProcedureDescription,TEXT,0,None,0
5,5,Charge,REAL,0,None,0
6,6,Payer,TEXT,0,None,0


In [111]:
# Use SQLite PRAGMA commands to inspect the structure of table LabTestResults
show_query("PRAGMA table_info(LabTestResults);")

SQL query:
PRAGMA table_info(LabTestResults);


,cid,name,type,notnull,dflt_value,pk
0,0,LabResultID,TEXT,0,None,1
1,1,VisitID,TEXT,1,None,0
2,2,PatientID,TEXT,1,None,0
3,3,LabTest,TEXT,0,None,0
4,4,ResultValue,TEXT,0,None,0
5,5,ResultDate,TEXT,0,None,0


## Query 5: Retrieve all female patients from the Patients table.

In [112]:
# Retrieve all female patients from the Patients table.

show_query("""
SELECT *
FROM Patients
Where Gender = 'Female';
""")

SQL query:
SELECT *
FROM Patients
Where Gender = 'Female';


,PatientID,Name,DOB,Gender,Address
0,P002,Maria Lopez,1990-11-03,Female,"892 North Ridge Avenue, Durham, NC 27701"
1,P004,Aisha Khan,1990-09-09,Female,"55 Harbor Point Road, Charlotte, NC 28202"


## Query 6: List all unique payers from the Visits table.

In [113]:
# List all unique payers from the Visits table.

show_query("""
SELECT DISTINCT Payer
FROM Visits
ORDER BY Payer;
""")

SQL query:
SELECT DISTINCT Payer
FROM Visits
ORDER BY Payer;


,Payer
0,Aetna
1,BlueCross
2,Cigna
3,Medicare


## Query 7: Display patient names along with their visit dates and procedure descriptions.

In [114]:
# Display patient names along with their visit dates and procedure descriptions. (use 'INNER JOIN')

show_query("""
SELECT p.Name, v.VisitDate, v.ProcedureDescription
FROM Patients p
INNER JOIN Visits v ON p.PatientID = v.PatientID;
""")

SQL query:
SELECT p.Name, v.VisitDate, v.ProcedureDescription
FROM Patients p
INNER JOIN Visits v ON p.PatientID = v.PatientID;


,Name,VisitDate,ProcedureDescription
0,John Smith,2024-01-12,Complete Blood Count (CBC)
1,Maria Lopez,2024-01-15,Office Visit – Established Patient
2,Tim Chen,2024-01-20,Venipuncture
3,John Smith,2024-02-01,Comprehensive Metabolic Panel (CMP)
4,Aisha Khan,2024-02-05,Rapid Strep Test


##Disclaimer:
This project is based on a lab exercise from the Foundations of Healthcare Data Analytics course on Coursera. Educational materials and original course content belong to their respective owners.

## End of Project